In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as sql_f

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("ActiveLearning")
    .getOrCreate()
)

# Obtenemos el SparkContext asociado a la SparkSession
sc = spark.sparkContext

## PREPROCESAMIENTO NECESARIO

In [2]:
# Lectura del dataframe
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ",")
    .csv("data/susy-10.csv")
)

In [3]:
# Convert basic columns
df = (
    df
    .withColumn("id_sample", sql_f.col("id_sample").cast("int"))
    .withColumn("label", sql_f.col("label").cast("int"))
)

# Convert all feature columns to float
feature_cols = [
    col_name
    for col_name in df.columns
    if col_name.startswith("feature_")
]

for col_name in feature_cols:
    df = df.withColumn(
        col_name,
        sql_f.col(col_name).cast("float")
    )

# Check schema
df.printSchema()

root
 |-- id_sample: integer (nullable = true)
 |-- label: integer (nullable = true)
 |-- feature_1: float (nullable = true)
 |-- feature_2: float (nullable = true)
 |-- feature_3: float (nullable = true)
 |-- feature_4: float (nullable = true)
 |-- feature_5: float (nullable = true)
 |-- feature_6: float (nullable = true)
 |-- feature_7: float (nullable = true)
 |-- feature_8: float (nullable = true)
 |-- feature_9: float (nullable = true)
 |-- feature_10: float (nullable = true)
 |-- feature_11: float (nullable = true)
 |-- feature_12: float (nullable = true)
 |-- feature_13: float (nullable = true)
 |-- feature_14: float (nullable = true)
 |-- feature_15: float (nullable = true)
 |-- feature_16: float (nullable = true)
 |-- feature_17: float (nullable = true)
 |-- feature_18: float (nullable = true)



In [4]:
df.show(1, truncate=False)

+---------+-----+----------+---------+----------+----------+---------+-----------+---------+---------+---------+----------+----------+----------+----------+----------+----------+----------+----------+----------+
|id_sample|label|feature_1 |feature_2|feature_3 |feature_4 |feature_5|feature_6  |feature_7|feature_8|feature_9|feature_10|feature_11|feature_12|feature_13|feature_14|feature_15|feature_16|feature_17|feature_18|
+---------+-----+----------+---------+----------+----------+---------+-----------+---------+---------+---------+----------+----------+----------+----------+----------+----------+----------+----------+----------+
|0        |1    |0.49275348|1.2934035|-0.5325326|0.47654477|1.4066684|-0.26715508|1.76809  |1.5441175|2.6541212|-1.2953297|0.40721557|1.2788798 |2.786904  |2.727609  |0.63572353|2.0354733 |1.3858824 |0.821922  |
+---------+-----+----------+---------+----------+----------+---------+-----------+---------+---------+---------+----------+----------+----------+-------

In [5]:
# Convertir las features a un vector con VectorAssembler para poder aplicar el modelo
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

# Se aplica sobre todo el conjunto de datos
df = assembler.transform(df)

# Se eliminan las columnas feature_cols
df = df.drop(*feature_cols)
df.show(1, truncate=False)

+---------+-----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|id_sample|label|features                                                                                                                                                                                                                                                                                                                                                 |
+---------+-----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
print(
    "Original DataFrame partitions:",
    df.rdd.getNumPartitions()
)

Original DataFrame partitions: 9


In [7]:
# Dividir en train/test
from config import TRAIN_TEST_SPLIT
from config import RANDOM_SEED
# División train/test
train_df, test_df = df.randomSplit(
    TRAIN_TEST_SPLIT,
    seed=RANDOM_SEED
)

In [8]:
# Seleccionar aleatoriamente conjunto etiquetado (L) y conjunto no etiquetado (U)
from config import INITIAL_LABELED_FRACTION

train_df = train_df.withColumn(
    "state",
    sql_f.when(
        sql_f.rand(RANDOM_SEED) < INITIAL_LABELED_FRACTION,
        "L"
    ).otherwise("U")
)

In [9]:
# Se almacena en cache tanto train_df como test_df
train_df.cache()
test_df.cache()

DataFrame[id_sample: int, label: int, features: vector]

In [10]:
# Se materializa la cache de train_df y se cuenta tanto labeled como unlabeled
stats = train_df.select(
    sql_f.count(sql_f.when(sql_f.col("state") == "L", True)).alias("labeled"),
    sql_f.count(sql_f.when(sql_f.col("state") == "U", True)).alias("unlabeled"),
).first()

# Extraer los valores
labeled_size = stats["labeled"]
unlabeled_size = stats["unlabeled"]
train_size = labeled_size + unlabeled_size
print(train_size)
print(labeled_size)
print(unlabeled_size)

70262
3490
66772


In [11]:
# Se materializa la cache de test_df
test_size = test_df.count()
print(test_size)

29738


In [12]:
print(
    "Original DataFrame partitions:",
    df.rdd.getNumPartitions()
)
print(
    "Train DataFrame partitions:",
    train_df.rdd.getNumPartitions()
)
print(
    "Test DataFrame partitions:",
    test_df.rdd.getNumPartitions()
)

Original DataFrame partitions: 9
Train DataFrame partitions: 9
Test DataFrame partitions: 9


## ENTRENAMIENTO INICIAL DEL CLASIFICADOR

In [13]:
# Entrenar clasificador LogisticRegresion de MLLlib para calcular probabilidades
from pyspark.ml.classification import LogisticRegression

# Crear clasificador
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability"
)

In [14]:
# Conjunto de datos etiquetado
labeled_df = train_df.filter(
    sql_f.col("state") == "L"
)

# Entrenar el modelo
lr_model = lr.fit(labeled_df)

In [15]:
print(
    "Labeled DataFrame partitions:",
    labeled_df.rdd.getNumPartitions()
)

Labeled DataFrame partitions: 9


**Evaluación inicial sobre test**

In [16]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Realizar predicciones
test_predictions = lr_model.transform(test_df)

# Accuracy
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = accuracy_evaluator.evaluate(test_predictions)

# AUC-ROC
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = auc_evaluator.evaluate(test_predictions)

print(f"Accuracy: {accuracy:.4f}")
print(f"AUC-ROC:  {auc:.4f}")

Accuracy: 0.7891
AUC-ROC:  0.8565


## INICIO DEL ALGORITMO ITERATIVO

**CÁLCULO DE LAS PREDICCIONES SOBRE EL CONJUNTO NO ETIQUETADO (U)**

Para clasificación binaria con Logistic Regression, una medida muy sencilla y habitual es la incertidumbre basada en la distancia a 0.5:
$$uncertainty=1−∣P(y=1)−0.5∣×2$$

Con esta formula la incertidumbre maxima se da en P(Y=1)=0.5 de modo que la incertidumbre es 1.

In [17]:
unlabeled_df = train_df.filter(
    sql_f.col("state") == "U"
)

unlabeled_predictions = lr_model.transform(unlabeled_df)
unlabeled_predictions.show(5, truncate=False)

+---------+-----+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+------------------------------------------+----------------------------------------+----------+
|id_sample|label|features                                                                                                                                                                                                                                                                                                                                                    |state|rawPrediction                             |probability                             |prediction|
+---------+-----+-----------------------------------------------

In [18]:
# Calculo de incertidumbre
from pyspark.ml.functions import vector_to_array

unlabeled_predictions = unlabeled_predictions.withColumn(
    "uncertainty",
    1 - 2 * sql_f.abs(
        vector_to_array("probability")[1] - 0.5
    )
)
unlabeled_predictions.show(5,truncate=False)

+---------+-----+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+------------------------------------------+----------------------------------------+----------+-------------------+
|id_sample|label|features                                                                                                                                                                                                                                                                                                                                                    |state|rawPrediction                             |probability                             |prediction|uncertainty        |
+---------+-----+-------

**Calculo del Top M con mayor incertidumbre**

In [19]:
from config import QUERY_BATCH_FRACTION
from config import UNCERTAINTY_CANDIDATES_MULTIPLIER


# Aqui hay que hacer unos prints informativos en los que
# se muestra el porcentaje de cada cosa y se entienda

# Tamaño del conjunto no etiquetado
# Esta variable es mejor obtenerla al inicio en el count
# una sola vez
unlabeled_size = unlabeled_predictions.count()

# No podemos pedir mas muestras al oráculo de las que quedan en U
# Hacer la operacion train_size * Query_BATCH_FRACTION al inicio
# almacenar en una variable llamada B y mostrar en configuracion
query_batch = min(
    int(train_size * QUERY_BATCH_FRACTION),
    unlabeled_size
)
p = min(1.0, (10 * query_batch) / unlabeled_size)
print(f"Training samples: {train_size}")
print(f"Unlabeled samples: {unlabeled_size}")
print(f"Query batch: {query_batch}")
print(f"Proporción de candidatos a seleccionar en fase de incertidumbre (p): {p*100}%")

Training samples: 70262
Unlabeled samples: 66772
Query batch: 702
Proporción de candidatos a seleccionar en fase de incertidumbre (p): 10.513388845623913%


In [20]:
print(
    "Unlabeled DataFrame partitions:",
    unlabeled_df.rdd.getNumPartitions()
)

Unlabeled DataFrame partitions: 9


In [21]:
quantile_target = 1.0 - p
print(quantile_target)

0.8948661115437608


In [22]:
# 3. Cálculo del umbral distribuido mediante approxQuantile (Greenwald-Khanna)
EPSILON = 0.001
threshold_val = unlabeled_predictions.stat.approxQuantile(
    "uncertainty", [quantile_target], EPSILON
)[0]

# 4. Filtrado distribuido directo (M_approx ≈ ALPHA * B)
uncertainty_candidates = unlabeled_predictions.select("id_sample","features","uncertainty").filter(sql_f.col("uncertainty") >= threshold_val)

In [23]:
print(
    "Uncertainty candidates DataFrame partitions:",
    uncertainty_candidates.rdd.getNumPartitions()
)

Uncertainty candidates DataFrame partitions: 9


In [24]:
uncertainty_candidates.count()

7087

In [25]:
uncertainty_candidates.show(5, truncate=False)

+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+
|id_sample|features                                                                                                                                                                                                                                                                                                                                                  |uncertainty       |
+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

**IMPLEMENTACIÓN DE DIVERSITY**

Para esta parte utilizaría una combinación de DataFrame + RDD interno. El DataFrame es la estructura principal, pero el algoritmo farthest-first necesita mantener un estado (min_distance) por candidato y ejecutar una función mapPartitions en los workers, por lo que un RDD es una forma mucho más natural y eficiente de implementarlo.

La optimización clave será no recalcular la distancia contra todos los seleccionados. Para cada candidato mantenemos:
$$D(x,S)=s∈Smin​d(x,s)$$
Cuando añadimos un nuevo seleccionado $s_{new}$, solamente necesitamos hacer:
$$Dnuevo​(x)=min(Danterior​(x),d(x,snew​))$$

En primer lugar se define una función para calcular la distancia entre dos vectores de caracteristicas. En este caso se usa la distancia al cuadrado.

Ahora definimos una función que hace el calculo de el candidato con distancia más alejada al conjunto seleccionado en cada partición.

In [26]:
def get_partition_farthest(partition):
    """Cada worker devuelve únicamente la muestra con mayor min_distance de su partición.
    # Cada fila de la partición contiene:
    (
        id_sample,      # row[0]
        features,       # row[1]
        min_distance    # row[2]
    )
    """
    farthest = None
    for row in partition:
        # row[0]: id_sample, row[1]: features, row[2]: min_distance
        if farthest is None or row[2] > farthest[2]:
            # Se guarda el mejor candidato hasta el momento si cumple la condicion
            farthest = row

    # Se devuelve el candidato más alejado de la partición  (si existe) con yield
    # porque mapPartitions() espera que la función produzca un iterador de resultados.
    if farthest is not None:
        yield farthest

La siguiente función es la parte fundamental de la optimización del Farthest-First. Su objetivo es actualizar la distancia mínima de cada candidato sin volver a calcular las distancias respecto a todos los elementos seleccionados anteriormente.

In [35]:
from pyspark.ml.linalg import Vectors
import numpy as np

def update_min_distance_old(partition, new_selected_id, new_selected_features):
    """ 
    partition es la partición del RDD que está procesando el worker:
    row[0] → id_sample
    row[1] → features
    row[2] → min_distance

    new_selected es el vector de características del nuevo candidato 
    contra el que queremos calcular la distancia y actualizarla en 
    row[2] si es necesario"""
    for row in partition:
        # Descartar la muestra recién elegida
        if row[0] == new_selected_id:
            continue

    # row[1] es DenseVector de tamaño 18, new_selected_features es DenseVector de tamaño 18
    dist = float(Vectors.squared_distance(row[1], new_selected_features))

    # Actualización del mínimo
    # Obtenemos la distancia minima entre la del candidato a las
    # anteriores instancias del conjunto y la de la nueva muestra actual
    yield (row[0], row[1], min(row[2], dist))

def update_min_distance(partition, new_id, new_features_np):
    """Actualiza distancias mínimas en la partición de forma vectorizada omitiendo el seleccionado."""
    rows = list(partition)
    if not rows:
        return

    # Excluir la muestra elegida
    filtered_rows = [r for r in rows if r[0] != new_id]
    if not filtered_rows:
        return

    ids = [r[0] for r in filtered_rows]

    # Convertir matriz de características a NumPy
    X = np.array([
        r[1].toArray() if hasattr(r[1], "toArray") else r[1]
        for r in filtered_rows
    ])
    current_min_dists = np.array([r[2] for r in filtered_rows], dtype=np.float64)

    # Distancia euclídea al cuadrado vectorizada (C-compiled)
    new_dists = np.sum((X - new_features_np) ** 2, axis=1)
    updated_min_dists = np.minimum(current_min_dists, new_dists)

    for i in range(len(filtered_rows)):
        yield (ids[i], filtered_rows[i][1], float(updated_min_dists[i]))

Ahora aplicamos el algoritmo iterativamente.

In [28]:
# 1. Obtener la fila completa con mayor incertidumbre
top_row = uncertainty_candidates.orderBy(
    sql_f.col("uncertainty").desc()
).first()

# 2. Extraer id_sample y features
first_id = top_row["id_sample"]
first_features = top_row["features"]
first_uncertainty = top_row["uncertainty"]
print(first_id)
print(first_features)
print(first_uncertainty)

32192
[1.024153709411621,-1.224080204963684,-1.6271849870681763,1.5975842475891113,-0.10291272401809692,0.2758602797985077,1.2615809440612793,-0.6999847888946533,1.8937885761260986,-0.8946791291236877,1.2686375379562378,1.3458539247512817,0.9414061307907104,1.5234140157699585,1.3366345167160034,1.4562387466430664,1.5216773748397827,0.002469239989295602]
0.9999994114683068


In [29]:
# Creamos la lista que almacena las instancias identificadas
# Esto hace que selected sea pequeño 
selected = [first_id]
print(selected)

[32192]


In [30]:
# ---------------------------------------------------------
# 2. Inicializar RDD de candidatos restantes
# ---------------------------------------------------------
# Se filtra para eliminar la muestra seleccionada
candidates_df_filter = uncertainty_candidates.filter(
    sql_f.col("id_sample") != first_id
)

In [31]:
print(
    "DataFrame partitions:",
    candidates_df_filter.rdd.getNumPartitions()
)

DataFrame partitions: 9


In [166]:
# Convertimos dataframe en RDD porque el algoritmo
# necesita operaciones como mappartitions()
# El RDD tiene la siguiente forma:
# id_sample -> row[0]
# features -> row[1]
# min_distance -> row[2]

# En esta iteracion inicial row[2] contiene la distancia minima
# al primer elemento seleccionado


# Creación del RDD inicial: (id_sample, features, min_distance)
candidates_rdd = candidates_df_filter.rdd.map(
    lambda row: (
        row["id_sample"],
        row["features"],
        Vectors.squared_distance(row["features"], first_features),
    )
)
# 2. Aplicar localCheckpoint() IN-PLACE (sin asignar a la variable)
candidates_rdd.localCheckpoint()

In [167]:
print(
    "candidates_rdd partitions:",
    candidates_rdd.getNumPartitions()
)

candidates_rdd partitions: 9


In [ ]:
# localCheckpoint elimina el DAG antiguo y libera linaje
#candidates_rdd = candidates_rdd.localCheckpoint()

In [168]:
# ---------------------------------------------
# Cada worker encuentra su candidato más lejano
# ---------------------------------------------
#
# El RDD candidates RDD está distribuido entre los
# workers 
# 
# Con mapPartitions se aplica la funcion 
# get_partition_farthest para que cada 
# particion calcule su candidato más lejano al
# conjunto de seleccionados actual

# Worker: Obtiene el más lejano local de cada partición
partition_candidates = candidates_rdd.mapPartitions(get_partition_farthest)

In [170]:
# ---------------------------------------------
# Driver obtiene el máximo global
# ---------------------------------------------
# En partition_candidates están los más alejados
# de cada partición por lo que el driver simplemente
# selecciona el más alejado de todos
farthest_candidate = partition_candidates.max(key=lambda row: row[2])

In [171]:
print(farthest_candidate)

(81486, DenseVector([5.2469, -1.9428, 1.5504, 6.8044, -0.5653, -0.2634, 0.7077, 0.3199, 0.4863, -0.3187, 6.5249, 2.1708, 0.2952, 1.5384, 6.5327, 1.4963, 0.3337, 0.0496]), 116.86080727595619)


In [172]:
# farthest_candidate tiene la siguiente estructura
# id_sample -> row[0]
# features ->  row[1]
# min_distance -> row[2]

# ---------------------------------------------
# Añadir a S
# ---------------------------------------------

# Añadir el id_sample
selected.append(farthest_candidate[0])
print(selected)

[32192, 81486]


In [174]:
farthest_id = farthest_candidate[0]
farthest_features = farthest_candidate[1]
# Broadcast de las features del nuevo seleccionado
new_selected_broadcast = sc.broadcast(farthest_features)

In [175]:
candidates_rdd = candidates_rdd.mapPartitions(
    lambda partition:
        update_min_distance(
            partition,
            farthest_id,
            new_selected_broadcast.value
        )
)

In [176]:
candidates_rdd.localCheckpoint()

In [177]:
# 5. Única limpieza manual necesaria: la variable broadcast
new_selected_broadcast.destroy()

print(
    f"Iteration {0 + 1}/{702} "
    f"- selected: {farthest_candidate[0]}"
)

Iteration 1/702 - selected: 81486


In [178]:
partition_candidates = candidates_rdd.mapPartitions(get_partition_farthest)

Realizamos la primera iteración para añadir otra muestra a selected

In [37]:
def farthest_first_selection(candidates_df, batch_size):
    """Selección Farthest-First distribuida y optimizada.

    candidates_df: DataFrame con [id_sample, features, uncertainty] batch_size:
    Número de muestras a seleccionar (B) sc: SparkContext activo
    """
    # ---------------------------------------------------------
    # 1. Primer seleccionado: Mayor incertidumbre
    # ---------------------------------------------------------
    first_candidate = candidates_df.orderBy(sql_f.col("uncertainty").desc()).first()
    first_id = first_candidate["id_sample"]
    first_features = first_candidate["features"]

    selected = [first_id]

    # ---------------------------------------------------------
    # 2. Inicializar RDD de candidatos restantes
    # ---------------------------------------------------------
    candidates_df_filter = candidates_df.filter(
        sql_f.col("id_sample") != first_id
    )

    # Creación del RDD inicial: (id_sample, features, min_distance)
    candidates_rdd = candidates_df_filter.rdd.map(
        lambda row: (
            row["id_sample"],
            row["features"],
            Vectors.squared_distance(row["features"], first_features),
        )
    )
    # Uso de local checkpoint
    candidates_rdd.localCheckpoint()

    # ---------------------------------------------------------
    # 3. Bucle Iterativo Farthest-First
    # ---------------------------------------------------------
    # Mantenemos una referencia al broadcast activo
    active_broadcast = None
    print("Iteracion inicial completada")
    for iteration in range(1, batch_size):

        # Worker: Obtiene el más lejano local de cada partición
        partition_candidates = candidates_rdd.mapPartitions(get_partition_farthest)

        # Driver: Obtiene el máximo global entre las respuestas de las particiones
        # CORREGIDO: row[2] contiene min_distance
        farthest_candidate = partition_candidates.max(key=lambda row: row[2])
        # 2. AHORA SÍ ES SEGURO: Una vez que .collect() terminó, destruimos
        #    el broadcast que se utilizó para calcular esa iteración

        if active_broadcast is not None:
            active_broadcast.destroy()

        farthest_id = farthest_candidate[0]
        farthest_features = farthest_candidate[1]

        # Se añade el id a selected
        selected.append(farthest_id)

        # Broadcast de las features del nuevo seleccionado
        active_broadcast = sc.broadcast(farthest_features)

        # Se actualizan las distancias respecto al nuevo seleccionado
        candidates_rdd = candidates_rdd.mapPartitions(
            lambda partition:
                update_min_distance(
                    partition,
                    farthest_id,
                    active_broadcast.value
                )
        )

        candidates_rdd.localCheckpoint()

        print(
            f"Iteración {iteration + 1}/{batch_size} - Seleccionado id:"
            f" {farthest_id}"
        )
    if active_broadcast is not None:
        active_broadcast.destroy()

    return selected

In [38]:
selected = farthest_first_selection(uncertainty_candidates, query_batch)

Iteracion inicial completada
Iteración 2/702 - Seleccionado id: 81486
Iteración 3/702 - Seleccionado id: 59919


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "c:\Users\david\anaconda3\envs\pymlspark_book\lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "c:\Users\david\anaconda3\envs\pymlspark_book\lib\site-packages\py4j\clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "c:\Users\david\anaconda3\envs\pymlspark_book\lib\socket.py", line 669, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 